# vLLM Support

[vLLM](https://github.com/vllm-project/vllm) is a popular library used for fast inference. By leveraging PagedAttention, dynamic batching, and Hugging Face model integration, vLLM makes inference more efficient and scalable for real-world applications.

Starting with `NNsight 0.4`, NNsight includes support for internal investigations of vLLM models.

## Setup

You will need to install `nnsight@vllmv1`, `vllm==0.12.0`, and `triton==3.5.0` to use vLLM with NNsight.

In [1]:
from IPython.display import clear_output
from pprint import pprint

try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    %pip install -U git+https://github.com/ndif-team/nnsight@vllmv1 triton==3.5.0 vllm==0.12.0 numpy==2.2.4
clear_output()

 Next, let's load in our NNsight-supported vLLM model. You can find vLLM-supported models [here](https://docs.vllm.ai/en/stable/models/supported_models.html). For this exercise, we will use GPT-2.

 Please note that vLLM models require a GPU to run.

In [2]:
from nnsight.modeling.vllm import VLLM

vllm = VLLM("meta-llama/Llama-3.1-8B", dispatch=True, tensor_parallel_size=2) # vLLM supports parallelism

print(vllm)

WARNING 12-03 17:13:49 [vllm.py:1322] Current vLLM config is not set.
INFO 12-03 17:13:49 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 12-03 17:13:49 [parallel_state.py:1200] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://127.0.0.1:47303 backend=gloo
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
WARNING 12-03 17:13:49 [vllm.py:1322] Current vLLM config is not set.
INFO 12-03 17:13:49 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=2048.
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of conne

/share/u/avery/nnsight/.venv/lib/python3.12/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:161: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
We recommend installing via `pip install torch-c-dlpack-ext`
  warnings.warn(


INFO 12-03 17:13:54 [cuda.py:411] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION']
INFO 12-03 17:13:57 [utils.py:253] non-default args: {'seed': None, 'tensor_parallel_size': 2, 'disable_log_stats': True, 'enforce_eager': True, 'worker_cls': 'nnsight.modeling.vllm.workers.GPUWorker.NNsightGPUWorker', 'model': 'meta-llama/Llama-3.1-8B'}
WARNING 12-03 17:13:57 [arg_utils.py:1175] `seed=None` is equivalent to `seed=0` in V1 Engine. You will no longer be allowed to pass `None` in v0.13.
WARNING 12-03 17:13:57 [arg_utils.py:1183] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 12-03 17:13:57 [model.py:637] Resolved architecture: LlamaForCausalLM
INFO 12-03 17:13:57 [model.py:1750] Using max model len 131072
INFO 12-03 17:13:57 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_toke

[rank0]:[W1203 17:14:14.789567107 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


Exception: WorkerProc initialization failed due to an exception in a background process. See stack trace for root cause.

## Interventions on vLLM models
We now have a vLLM model that runs with `nnsight`. Let's try applying some interventions on it.

In [ ]:
neurons = [394, 5490, 8929]
prompt = "The truth is the"

mlp = vllm.model.layers[16].mlp.down_proj

with vllm.trace(prompt, remote=False):
    mlp.input = mlp.input.clone()
    mlp.input[-1, neurons] = 10 # no batch dimension
    out = vllm.output.save()
    last = out[:, -1].argmax()  # returns a tensor
    prediction = vllm.tokenizer.decode(last).save()

print(f"Prediction with vLLM: '{prediction}'")


Note that because of differences in default inference settings, and other implementation details that may be specific to your runtime stack, results may differ compared to Huggingface Transformers, even in the same intervention!

In [ ]:
# Use the HuggingFace transformers backend for comparison
from nnsight import LanguageModel

neurons = [394, 5490, 8929]
prompt = "The truth is the"

lm = LanguageModel(MODEL_ID, dispatch=True, device_map="auto") # transformers supports device_map
mlp = lm.model.layers[16].mlp.down_proj

with lm.trace(prompt, remote=False):
    mlp.input[:, -1, neurons] = 10                # batch dimension
    out = lm.output.save()
    last = out["logits"][:, -1].argmax()          # dict of tensors
    prediction = lm.tokenizer.decode(last).save()

print(f"Prediction with transformers: '{prediction}'")

We've successfully performed an intervention on our vLLM model!

## Sampled Token Traceability
vLLM provides functionality to configure how each sequence samples its next token. Here's an example of how you can trace token sampling operations with the nnsight VLLM wrapper.

In [ ]:
with vllm.trace("Madison Square Garden is located in the city of", temperature=0.8, top_p=0.95, max_tokens=3) as tracer:
    samples = list().save()
    logits = list().save()

    for ii in range(3):
        tracer.next()
        samples.append(vllm.samples.output)
        tracer.next()
        logits.append(vllm.logits.output)
    samples.save()
    logits.save()

pprint(samples)
pprint(logits) # different than samples with current sampling parameters

<details>
<summary>
Note: gradients are not supported with vLLM
</summary>

vLLM speeds up inference through its paged attention mechanism. This means that accessing gradients and backward passes are not supported for vLLM models. As such, calling gradient operations when using `nnsight` vLLM wrappers will throw an error.
</details>

## Other features

### Intervening on generated token iterations with .all() and .iter[]
NNSight supports iteration via `all()` and `iter()`

In [ ]:
with vllm.trace("Hello World!", max_tokens=10) as tracer:
    outputs = list().save()

    # will iterate over all 10 tokens
    with tracer.all():
        out = vllm.output[:, -1]
        outputs.append(out)

print(len(outputs))
print("".join([vllm.tokenizer.decode(output.argmax()) for output in outputs]))

In [ ]:
prompt = 'The Eiffel Tower is in the city of'
mlp = vllm.model.layers[16].mlp.down_proj
n_new_tokens = 50

with vllm.trace(prompt, max_tokens=n_new_tokens) as tracer:
    hidden_states = list().save() # Initialize & .save() list

    # Call .iter() to apply intervention to specific new tokens
    with tracer.iter[2:5]:

        # Apply intervention - set to zero
        mlp.input = mlp.input.clone()
        mlp.input[-1] = 0

        # Append hidden state post-intervention
        hidden_states.append(mlp.input) # no need to call .save

print("Hidden state length: ",len(hidden_states))
pprint(hidden_states)